In [70]:
import sys
print(sys.executable)


c:\Users\elois\OneDrive\Documents\GitHub\Proyecto2_ML\.venv\Scripts\python.exe


# **0. LIBRERÍAS Y CONFIGURACIÓN**

In [71]:
import pandas as pd
import numpy as np
import plotly.express as px

# Preprocesamiento y Modelado
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.pipeline import Pipeline as SklearnPipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import PowerTransformer
# modelos

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from lightgbm import LGBMRegressor
from sklearn.metrics import precision_recall_curve, confusion_matrix


# Métricas
import optuna
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score



# **1. Carga de datos procesados y exploración inicial**

In [72]:
df = pd.read_parquet("../data/processed/dataset.parquet")



# **Feature engineering**

In [73]:
# Filtro de precios entre 500k y 50M, filtro que se había creado en el EDA toca volver a crearlo aquí para que el modelo no se vea afectado por outliers
df_filtrado = df[(df["Precio"] >= 500_000) & (df["Precio"] <= 50_000_000) & (df["Área Construida (m2)"] <= 2500)].copy()


In [74]:
# Definimos nuevamente piso_cat que quedó en el EDA

def piso_cat(x):
    if pd.isna(x):
        return "missing"
    elif x == 1:
        return "1"
    elif x == 2:
        return "2"
    else:
        return "3+"

df_filtrado["Piso_cat"] = df_filtrado["Piso N°"].apply(piso_cat)


# **Split de datos (train/test)**

### Separación de variables predictoras (X) y variable objetivo (y, "Precio")

### <span style="color: #dc2626;">!!! **Importante: comentarios para recordar**</span>


1. voy a usar Barrio_group como parte de las features, pero la idea es que luego podamos definir realmente cuales van a ser las variables que vamos a usar despues de que toda la limpieza y análisis haya terminado, estoy usando el que hace que vaya a "otros"
2. también estoy usando piso_cat
3. pareciera que para poder usar optuna, es mejor hacer algo como train, test y validation porque optuna corremos el riesgo de ajustar el modelo al test indirectamente, la idea es que  Optuna compare configuraciones sin “mirar” el test final, y eso es algo que no hicimos en el trabajo de la profe camila

70% train: el modelo aprende
15% validation : Optuna prueba distintas combinaciones y decide cuáles son mejores
15% test: una sola vez al final para medir el desempeño real

4. tenemos que definir que métrica nos importa mas para nuestros modelos entre MAE, RMSE y R2


In [75]:
df_filtrado.columns

Index(['ID', 'Barrio', 'Tipo de Inmueble', 'Estado', 'Antigüedad',
       'Área Construida (m2)', 'Área Privada (m2)', 'Estrato', 'Baños',
       'Habitaciones', 'Parqueaderos', 'Piso N°', 'URL', 'Precio',
       'Barrio_clean', 'Barrio_group', 'Piso_cat'],
      dtype='str')

In [76]:
numeric_features = [
    "Área Construida (m2)",
    "Área Privada (m2)",
    "Estrato",
    "Baños",
    "Habitaciones",
    "Parqueaderos",
]

categorical_features = [
    "Tipo de Inmueble",
    "Estado",
    "Antigüedad",
    "Barrio_group",
    "Piso_cat",
]

features = numeric_features + categorical_features



In [77]:
# Separación de variables predictoras (X) y variable objetivo (y, "Precio")

X = df_filtrado[features]
y = df_filtrado["Precio"]

## 70% train, 30% temporal test

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.30,
    random_state=42,
)  

# Del 30% temporal, mitad validation y mitad test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.50,
    random_state=42
)

# Checamos el shape de las variables de entreno y prueba X, Y
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_val:", X_val.shape)
print("y_val:", y_val.shape)
print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

X_train: (3723, 11)
y_train: (3723,)
X_val: (798, 11)
y_val: (798,)
X_test: (798, 11)
y_test: (798,)


## **Pipeline: preprocesamiento + modelo (evitar data leakage)**

In [78]:
df_filtrado.info()

<class 'pandas.DataFrame'>
Index: 5319 entries, 0 to 5627
Data columns (total 17 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   ID                    5319 non-null   str    
 1   Barrio                5319 non-null   str    
 2   Tipo de Inmueble      5319 non-null   str    
 3   Estado                5319 non-null   str    
 4   Antigüedad            4037 non-null   str    
 5   Área Construida (m2)  5319 non-null   float64
 6   Área Privada (m2)     5319 non-null   float64
 7   Estrato               5270 non-null   float64
 8   Baños                 5278 non-null   float64
 9   Habitaciones          5231 non-null   float64
 10  Parqueaderos          5319 non-null   int64  
 11  Piso N°               2850 non-null   float64
 12  URL                   5319 non-null   str    
 13  Precio                5319 non-null   int64  
 14  Barrio_clean          5319 non-null   str    
 15  Barrio_group          5319 non-null  

### **Pipelines y transformaciones**

Modelos: 

1. LinearRegression
2. RandomForestRegressor
3. LGBMRegressor

- `PowerTransformer` para normalizar la distribución del Precio, usamos `Yeo-Johnson` porque a pesar de que nuestra variable `Precio` ya tiene valores positivos y podríamos usar `Box-Cox`, nos pareció mejor un metodo que fuera tan estricto, es mas flexible, y aunque ambos buscan redcir la asimetría y estabilizar la varianza, Box-Cox es estricamente mayores a 0, y queríamos una transformación mas segura y fácil de integrar en la pipeline

ademas, es mas flexible que hacer escala logarítimca `np.log1p` sobre nuestra y "precio" porque se adapta a los datos en vez de asumir siempre logartimo.

- PowerTransformer(y)  →  normaliza la distribución del PRECIO 
- RobustScaler(X)      →  escala numeric_features siendo robusto a outliers solo para LinearRegression porque en los de arboles el escalado no influye

Para el modelo lineal (`LinearRegression`) se usa **`RobustScaler`** en lugar de `StandardScaler`.

| Scaler | Fórmula | Problema |
|---|---|---|
| `StandardScaler` | `z = (x − media) / std` | La media y std se ven jaladas por outliers |
| `RobustScaler` | `z = (x − mediana) / IQR` | La mediana y el IQR son resistentes a outliers |

IQR (Rango Intercuartílico)** es la distancia entre el percentil 25 (Q1) y el percentil 75 (Q3).
Cubre el **50% central de los datos**, ignorando los extremos al calcular la escala.

El dataset mezcla habitaciones en arriendo (~20–40 m²) con apartamentos en El Poblado (~200–480 m²).
Con `StandardScaler`, esos valores extremos jalan la media hacia arriba y comprimen el resto.
Con `RobustScaler`, los valores centrales se escalan correctamente sin importar los extremos.

In [79]:
areas = ["Área Construida (m2)", "Área Privada (m2)"]

df_filtrado[areas].describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]).T


,count,mean,std,min,1%,5%,25%,50%,75%,95%,99%,max
Área Construida (m2),5319.0,106.699818,104.442245,1.0,20.0,35.0,60.0,80.0,115.0,283.2,500.0,2416.0
Área Privada (m2),5319.0,112.156294,248.075961,1.0,20.0,35.0,60.0,79.0,115.0,285.3,508.2,14770.0


<span style="color: red;"><strong>!!!Observación</strong></span>

Hay que filtrar esa area tan grandota de 55.200, siento que es un error, puede que sea ese edificio que dice Simón con el de 25 baños?

fijense que también, la media esta en 414 m2 pero el 75% de los datos esta en 120,2 y el 95% en 400m2 o menos, si hacemos simplemente `standardScaler` con esa media de 414m2 y std de 2251, lo que sucedería es que un apto de unos 80m2 que es lo normal quedé que si en -0.15, casi igual que uno de 20m2

se que vamos a quitar esos extremos directamente, pero por si acaso uso robustscaler y voy a filtrar hasta 4000 m2, también podemos pensar si realmente queremos arriendos medellín ams residencial o esas casas campestres





In [80]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=("Área Construida (m2)", "Área Privada (m2)")
)

fig.add_trace(
    go.Box(
        x=df_filtrado["Área Construida (m2)"],
        name="Área Construida",
        marker=dict(color="#0f766e"),
        fillcolor="rgba(15, 118, 110, 0.35)",
        line=dict(color="#0f766e"),
        boxmean=True,
        boxpoints="outliers",
        hovertemplate="Área construida: %{x:.2f} m²<extra></extra>"
    ),
    row=1,
    col=1
)

fig.add_trace(
    go.Box(
        x=df_filtrado["Área Privada (m2)"],
        name="Área Privada",
        marker=dict(color="#b45309"),
        fillcolor="rgba(180, 83, 9, 0.35)",
        line=dict(color="#b45309"),
        boxmean=True,
        boxpoints="outliers",
        hovertemplate="Área privada: %{x:.2f} m²<extra></extra>"
    ),
    row=1,
    col=2
)

fig.update_layout(
    title="Distribución y valores extremos en las variables de área",
    template="plotly_white",
    showlegend=False,
    width=1050,
    height=450,
    font=dict(size=13),
    margin=dict(t=70, l=40, r=40, b=40)
)

fig.update_xaxes(title_text="Metros cuadrados", row=1, col=1)
fig.update_xaxes(title_text="Metros cuadrados", row=1, col=2)

fig.show()


In [81]:
# Mira qué hay entre 500 y 4000
df[(df["Área Construida (m2)"] > 500) & 
   (df["Área Construida (m2)"] <= 4000)][["Barrio", "Tipo de Inmueble", "Área Construida (m2)", "Precio"]].sort_values("Área Construida (m2)", ascending=False).head(20)

,Barrio,Tipo de Inmueble,Área Construida (m2),Precio
216,Castilla,Apartamento,4000.0,950000
1483,Buenos aires,Apartamento,4000.0,1150000
255,Pedregal,Apartamento,4000.0,800000
218,Girardot,Apartaestudio,4000.0,1100000
207,Castilla,Apartamento,4000.0,1000000
706,Boston,Apartamento,3900.0,830000
705,Boston,Apartamento,3900.0,830000
704,Boston,Apartamento,3900.0,830000
1330,Buenos aires,Apartamento,3800.0,1200000
2173,Centro,Apartaestudio,3700.0,1100000


In [82]:
# Transformers

# ---------------------------Transformadores---------------------------

numeric_transformer_lr = Pipeline(
    steps=[

        
          ]
)